In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# -------------------------
# 1. Define State
# -------------------------
class QuadraticState(TypedDict):
    a: float
    b: float
    c: float
    equation: str
    discriminant: float
    result: str


# -------------------------
# 2. show_equation node
# -------------------------
def show_equation(state: QuadraticState):
    a = state["a"]
    b = state["b"]
    c = state["c"]

    equation = f"{a}x² + {b}x + {c} = 0"

    print("Equation:", equation)

    return {
        "equation": equation
    }


# -------------------------
# 3. calculate_discriminant
# -------------------------
def calculate_discriminant(state: QuadraticState):
    a = state["a"]
    b = state["b"]
    c = state["c"]

    d = b**2 - 4*a*c

    print("Discriminant:", d)

    return {
        "discriminant": d
    }


# -------------------------
# 4. Conditional Router
# -------------------------
def check_discriminant(state: QuadraticState):

    d = state["discriminant"]

    if d < 0:
        return "no_real_roots"

    elif d == 0:
        return "repeated_roots"

    else:
        return "real_roots"


# -------------------------
# 5. no_real_roots node
# -------------------------
def no_real_roots(state: QuadraticState):

    print("No real roots")

    return {
        "result": "No real roots"
    }


# -------------------------
# 6. repeated_roots node
# -------------------------
def repeated_roots(state: QuadraticState):

    print("Repeated roots")

    return {
        "result": "Repeated roots"
    }


# -------------------------
# 7. real_roots node
# -------------------------
def real_roots(state: QuadraticState):

    print("Two real roots")

    return {
        "result": "Two real roots"
    }


# -------------------------
# 8. Create Graph
# -------------------------
builder = StateGraph(QuadraticState)


# Add nodes
builder.add_node("show_equation", show_equation)
builder.add_node("calculate_discriminant", calculate_discriminant)
builder.add_node("no_real_roots", no_real_roots)
builder.add_node("repeated_roots", repeated_roots)
builder.add_node("real_roots", real_roots)


# -------------------------
# 9. Normal Edges
# -------------------------

builder.add_edge(START, "show_equation")

builder.add_edge(
    "show_equation",
    "calculate_discriminant"
)


# -------------------------
# 10. Conditional Edge
# -------------------------

builder.add_conditional_edges(
    "calculate_discriminant",
    check_discriminant,
    {
        "no_real_roots": "no_real_roots",
        "repeated_roots": "repeated_roots",
        "real_roots": "real_roots",
    }
)


# -------------------------
# 11. Connect all branches to END
# -------------------------

builder.add_edge("no_real_roots", END)

builder.add_edge("repeated_roots", END)

builder.add_edge("real_roots", END)


# -------------------------
# 12. Compile
# -------------------------

app = builder.compile()

In [2]:
result = app.invoke({
    "a": 1,
    "b": 2,
    "c": 5
})

print(result)

Equation: 1x² + 2x + 5 = 0
Discriminant: -16
No real roots
{'a': 1, 'b': 2, 'c': 5, 'equation': '1x² + 2x + 5 = 0', 'discriminant': -16, 'result': 'No real roots'}
